<a href="https://colab.research.google.com/github/bharath007-su/aiforcs_demo/blob/main/Lab-1_CS004.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Tiny next-word predictor in PyTorch (works in Google Colab as-is).
# Pipeline: text -> tokens -> embeddings -> neural net -> scores -> softmax -> sample
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# ---- 1. A very small training corpus (cybersecurity flavoured) -------------
text = """
the firewall blocks the bad packet . the firewall allows the good packet .
the analyst reads the log . the analyst finds the attack . the attack hits the server .
the server logs the attack . the model detects the attack . the model flags the bad packet .
the analyst trusts the model . the model reads the log . the firewall logs the packet .
""".split()

# ---- 2. Tokens: map each word to a number ----------------------------------
vocab = sorted(set(text))
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
V = len(vocab)
print("Vocabulary size:", V)

# ---- 3. Training pairs: 2 previous words -> next word ----------------------
CONTEXT = 2
X, Y = [], []
for i in range(len(text) - CONTEXT):
    X.append([stoi[w] for w in text[i:i + CONTEXT]])
    Y.append(stoi[text[i + CONTEXT]])
X, Y = torch.tensor(X), torch.tensor(Y)
print("Training examples:", len(X))


# ---- 4. The model -----------------------------------------------------------
class NextWord(nn.Module):
    def __init__(self, vocab_size, emb_dim=16, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)           # tokens -> vectors
        self.net = nn.Sequential(                              # stand-in for transformer layers
            nn.Linear(CONTEXT * emb_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, vocab_size))                     # one score per word

    def forward(self, x):
        e = self.emb(x).flatten(1)       # (batch, CONTEXT * emb_dim)
        return self.net(e)               # logits = raw scores


model = NextWord(V)
opt = torch.optim.Adam(model.parameters(), lr=0.01)

# ---- 5. Training: guess the next word, measure the error, adjust ------------
for epoch in range(301):
    logits = model(X)
    loss = F.cross_entropy(logits, Y)    # softmax + "how surprised was the model"
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.3f}")


# ---- 6. Predict: scores -> softmax -> probabilities -------------------------
def next_word_probs(words, top=5):
    x = torch.tensor([[stoi[w] for w in words]])
    with torch.no_grad():
        probs = F.softmax(model(x), dim=-1)[0]
    p, idx = probs.topk(top)
    return [(itos[i.item()], round(v.item(), 3)) for v, i in zip(p, idx)]


print("\nAfter 'the firewall':", next_word_probs(["the", "firewall"]))
print("After 'the analyst' :", next_word_probs(["the", "analyst"]))
print("After 'the model'   :", next_word_probs(["the", "model"]))


# ---- 7. Generate text one word at a time, with temperature -----------------
def generate(start, n_words=8, temperature=1.0):
    words = list(start)
    for _ in range(n_words):
        x = torch.tensor([[stoi[w] for w in words[-CONTEXT:]]])
        with torch.no_grad():
            logits = model(x)[0] / temperature      # low T = sharper, high T = flatter
        probs = F.softmax(logits, dim=-1)
        words.append(itos[torch.multinomial(probs, 1).item()])
    return " ".join(words)


for t in (0.3, 1.0, 2.0):
    print(f"\nTemperature {t}:")
    print(generate(["the", "analyst"], temperature=t))

Vocabulary size: 20
Training examples: 67
epoch   0  loss 3.026
epoch  50  loss 0.482
epoch 100  loss 0.481
epoch 150  loss 0.481
epoch 200  loss 0.481
epoch 250  loss 0.481
epoch 300  loss 0.481

After 'the firewall': [('logs', 0.333), ('blocks', 0.333), ('allows', 0.333), ('.', 0.0), ('packet', 0.0)]
After 'the analyst' : [('trusts', 0.333), ('finds', 0.333), ('reads', 0.333), ('packet', 0.0), ('the', 0.0)]
After 'the model'   : [('.', 0.25), ('reads', 0.25), ('detects', 0.25), ('flags', 0.25), ('the', 0.0)]

Temperature 0.3:
the analyst reads the log . the analyst trusts the

Temperature 1.0:
the analyst reads the log . the server logs the

Temperature 2.0:
the analyst trusts the model reads the log . the
